# Notebook 2: Logit Lens
**Kurs:** Mechanistic Interpretability  
**Modell:** EleutherAI/pythia-410m  
**Ziel:** Vorhersagen nach jeder Schicht visualisieren.

> **Aufgabe:** Implementiere die Zellen schrittweise. Hilfreiche Dokumentation:
> - HuggingFace `output_hidden_states`: https://huggingface.co/docs/transformers/main_classes/output#transformers.modeling_outputs.BaseModelOutputWithPast
> - PyTorch Softmax: https://pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html
> - Matplotlib Imshow: https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.imshow.html

## Hintergrund: Was ist der Logit Lens?

Der **Logit Lens** (nostalgebraist, 2020) ist eine Technik, um zu sehen, was ein Transformer "nach jeder Schicht denkt".

**Kernidee:** Transformer haben einen **Residual Stream** — ein Informationskanal, der durch alle Schichten fließt. Jede Schicht *addiert* ihren Beitrag.

Da der finale Output die letzte Projektion des Residual Streams auf den Vokabular-Raum ist, können wir dieselbe **Unembed-Matrix W_U** auch auf frühere Hidden States anwenden:

```
Logit Lens bei Schicht L: logits_L = hidden_state_L @ W_U.T
```

So sehen wir, wie sich die Vorhersage über die Schichten herausbildet.

**Frage:** In welcher Schicht "weiß" das Modell die richtige Antwort?

## 1. Setup und Modell laden

In [ ]:
# TODO: Importiere torch, numpy, matplotlib
# TODO: Lade Tokenizer und Modell (wie in Notebook 1)
# 
# MODEL_NAME = "EleutherAI/pythia-410m"
# Fallback: MODEL_NAME = "gpt2-medium"  (bei gpt2-medium: model.lm_head.weight statt model.embed_out.weight)

## 2. Forward Pass mit Hidden States

In [ ]:
clean_prompt     = "The capital of France is"
corrupted_prompt = "The capital of Xyzzy is"

# TODO: Tokenisiere beide Prompts
# TODO: Führe einen Forward Pass mit output_hidden_states=True durch
#
#   with torch.no_grad():
#       outputs = model(**inputs, output_hidden_states=True)
#
# TODO: Analysiere outputs.hidden_states:
#   - Wie viele Hidden States gibt es? (Tipp: num_layers + 1, Index 0 = Embedding)
#   - Was ist die Shape jedes Hidden State?
#
# Dokumentation: https://huggingface.co/docs/transformers/main_classes/output

## 3. Logit Lens Funktion implementieren

In [ ]:
# Die Unembed-Matrix für Pythia:
#   unembed = model.embed_out.weight  # Shape: (vocab_size, hidden_size)
# Für GPT-2 Medium:
#   unembed = model.lm_head.weight

# TODO: Implementiere die Funktion logit_lens(hidden_state, top_k=5):
#   1. Nimm den Hidden State der LETZTEN Token-Position: hidden_state[0, -1, :]
#   2. Berechne die Logits: last_token_hs @ unembed.T
#   3. Wende Softmax an: torch.softmax(logits, dim=-1)
#   4. Berechne Top-K: torch.topk(probs, top_k)
#   5. Dekodiere die Token-IDs: tokenizer.decode([id])
#   6. Gib eine Liste von (token_string, probability) Tupeln zurück
#
# Hinweis: torch.Tensor.detach() löst den Tensor vom Berechnungsgraphen
# Hinweis: @ ist der Matrixmultiplikationsoperator in Python/PyTorch

## 4. Logit Lens über alle Schichten berechnen

In [ ]:
# TODO: Iteriere über alle Hidden States (outputs.hidden_states)
#   und rufe logit_lens() für jeden auf.
#
# TODO: Speichere die Ergebnisse in einer Liste und gib eine Tabelle aus:
#   - Layer | Top-1 Token (Clean) | P(Clean) | Top-1 Token (Corrupt) | P(Corrupt)
#
# Frage: Ab welcher Schicht ist " Paris" unter den Top-1-Vorhersagen?

## 5. Wahrscheinlichkeit des richtigen Tokens pro Schicht

In [ ]:
# TODO: Implementiere get_token_prob_per_layer(outputs, token_id):
#   Gibt für jeden Layer die Wahrscheinlichkeit eines bestimmten Tokens zurück.
#
# correct_token = " Paris"
# correct_tok_id = tokenizer.encode(correct_token)[0]
#
# TODO: Rufe die Funktion für clean und corrupted auf.
# TODO: Erstelle einen Linienchart (plt.plot) mit:
#   - X-Achse: Schicht-Index (0 = Embedding)
#   - Y-Achse: P(" Paris")
#   - Eine Linie für clean, eine für corrupted
#   - Horizontale Linie bei 0.5 als Referenz
#
# Dokumentation plt.plot: https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html

## 6. Heatmap: Top-5 Token je Schicht

In [ ]:
# BONUS: Erstelle eine Heatmap mit plt.imshow():
#   - Y-Achse: Schichten (0 bis n_layers)
#   - X-Achse: Top-K Rang (0 bis 4)
#   - Farbe: Wahrscheinlichkeit des Tokens
#   - Text in jeder Zelle: Token-String (ax.text(...))
#
# Tipp: Erstelle zuerst eine Matrix (n_layers × top_k) mit Wahrscheinlichkeiten
# Tipp: plt.imshow(matrix, cmap="Blues") erstellt die Heatmap
#
# Dokumentation Imshow:
#   https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.imshow.html

## 7. Reflexionsfragen

In [ ]:
# Beantworte folgende Fragen als Kommentare oder Markdown-Zellen:
#
# 1. In welcher Schicht taucht " Paris" erstmals in den Top-5 auf (Clean)?
# 2. Was passiert bei den frühen Schichten (0-3)?
# 3. Warum bleibt P(" Paris") für den Corrupted-Prompt durchgehend niedrig?
# 4. Was zeigt uns der Logit Lens über den Residual Stream?
#
# Hinweis: Es gibt keine "richtige" Antwort — beschreibe, was du beobachtest.